# Day 2-01：画像を分類する

CIFAR-10の画像を10種類に分類するモデルを学習し、保存します。

## 実行前の準備

**`DRIVE_PROJECT`を自分のDrive内の教材フォルダに合わせてください。**
ローカルでは`notebooks/day2/`または教材ルートから実行します。ColabではGPUランタイムを選びます。

In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parent.parent
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)

## 1. データ

入力はRGB画像、正解は0〜9のクラス番号です。画像テンソルの形は`[枚数, 色のチャンネル, 高さ, 幅]`です。画素値は0〜1とします。

In [ ]:
from jkj1a.classification import load_cifar10
from jkj1a.classification import CIFAR_CLASSES
import torch
from torch import nn

torch.manual_seed(0)  # シード値を固定（毎回同じ結果が出るように）

# データを読み込む
train_loader, test_loader = load_cifar10(batch_size=128)

# 1バッチを取り出して形を確認する
x, y = next(iter(train_loader))
print("画像:", x.shape, "正解:", y.shape)
print(CIFAR_CLASSES)


In [ ]:
import matplotlib.pyplot as plt

# 6枚の画像と正解クラスを並べて表示する
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5), constrained_layout=True)

for ax, image, label in zip(axes, x, y):
    ax.imshow(image.permute(1, 2, 0))  # 描画用に(色, 高さ, 幅)から(高さ, 幅, 色)へ並べ替える
    ax.set_title(CIFAR_CLASSES[label.item()])
    ax.axis("off")

plt.show()
plt.close(fig)

### 演習：データの枚数

`test_loader`を使い、テスト画像の総数と、各クラスに属する画像の枚数を数えて表示してください。

In [ ]:
# ここにコードを書く

## 2. 分類モデル

CNNは画像の局所的な特徴を取り出し、最後に10クラス分のスコア（logits）を出力します。`argmax(dim=1)`で各画像の予測クラスを得ます。

In [ ]:
from jkj1a.classification import CIFARClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # cuda（GPU)が使用可能か確認．なけらばCPU
model = CIFARClassifier().to(device)  # モデルをデバイス（GPU or CPU）に載せる
print(model)

# 画像ごとのクラススコアを計算する
logits = model(x.to(device))  # .to(device)でデータもモデルと同じデバイスに載せるのを忘れない！
print("スコア:", logits.shape)
print("予測:", logits.argmax(dim=1)[:8])
print("正解:", y[:8])

### 演習：確信度を計算する

logitsは正規化前のスコアで、負の値も取り、合計は1とは限りません。logitsそのものは確率ではありません。

各画像の10クラスのlogitsにsoftmaxを適用して、合計が1になるクラスごとの確率（確信度; confidence score）を計算してください。最初の画像について、10個の値、その合計、最も確信度が高いクラスとその値を表示してください。

softmaxは数式では以下のような関数です。

$$
\operatorname{softmax}(\mathbf{z})_i
= \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
\qquad (i = 1, \ldots, K)
$$

ここで、$z_i$はクラス$i$のlogit、$K$はクラス数です。今回は$K=10$です。

In [ ]:
# ここにコードを書く
confidence = ...

softmaxの関数の適用により得られた値（確信度）は確率となっています。つまり、
- 各$i$について，非負の値になる
- $i=1, 2, \ldots, K$にわたる合計は1になる

となっています。

In [ ]:
print("非負の値になっているか:", confidence >= 0)
print("合計が1になっているか:", confidence.sum(dim=1) == 1)

## 3. 損失関数と学習

Day1では数値を予測しました。今回は正しいクラスのスコアを他のクラスより高くするように学習します。`CrossEntropyLoss`にはsoftmaxを適用する前のlogitsと、整数の正解ラベルを渡します。

### 演習：学習の処理を実装する

`ClassificationTrainer`の`train_step()`にある2か所の`pass`を、次の処理に置き換えてください。

1. `self.model`で入力`x`の予測を求め、`self.loss_fn`で正解`y`に対する損失`loss`を計算する。
2. 損失から勾配を計算し、`self.optimizer`でモデルの重みを更新する。

Day1で行ったTrainerの実装と比較し，相違点を観察すると理解が進むと思います


In [ ]:
class ClassificationTrainer:
    def __init__(self, model, data, loss_fn, optimizer):
        self.model = model
        self.data = data
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, n_epochs=5):

        for epoch in range(n_epochs):
            loss = self.train_step()
            print(f"Epoch {epoch + 1}, Loss: {loss:.4f}", flush=True)

    def train_step(self):

        # 学習モードに切り替える
        self.model.train()
        device = next(self.model.parameters()).device

        # 全画像の損失と枚数を集計する
        loss_total = 0
        n_samples = 0

        for x, y in self.data:
            # データをデバイスに載せる
            x, y = x.to(device), y.to(device)

            # 前のバッチの勾配を消す
            self.optimizer.zero_grad()

            # 予測と損失を計算する
            pass

            # 勾配を計算し、重みを更新する
            pass

            # バッチの枚数を考慮して損失を足す
            loss_total += loss.item() * len(y)
            n_samples += len(y)

        # 1画像あたりの平均損失を返す
        return loss_total / n_samples

    def evaluate(self, test_loader):
        # 評価モードに切り替える
        self.model.eval()

        # 評価では勾配を計算しない
        with torch.no_grad():
            # 分類精度を計算し、返す処理を書く
            pass


下のセルでtrainerを作成し、5 epoch学習してください。

In [ ]:
# 損失関数と、重みを更新する方法を決める
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.03, momentum=0.9)

# 学習に必要なものをTrainerに渡す
trainer = ClassificationTrainer(model, train_loader, loss_fn, optimizer)

trainer.train(n_epochs=5)


### 演習：学習履歴を記録する

上で定義した`ClassificationTrainer`に`self.history`という属性を追加し、各epochの平均損失を記録するように修正して下さい。

修正したクラスからモデル・optimizer・trainerを作り直して学習し、epochを横軸、平均損失を縦軸にしたグラフを描いてください。

In [ ]:
# ここにコードを書く

## 4. 評価

テストデータはパラメータの更新に使いません。まず、テスト画像に対する予測クラスと正解を見比べます。


In [ ]:
from jkj1a.classification import show_predictions

x, y = next(iter(test_loader))  # テストデータの最初のバッチを取り出す

# 画像ごとに、予測クラス・確信度・正解を表示する
show_predictions(model, x, CIFAR_CLASSES, y=y)

### 演習：分類精度を評価する

`ClassificationTrainer`の`evaluate(self, test_loader)`の空欄を実装してください。`test_loader`に含まれる全画像について、正しく分類できた画像の割合（分類精度）を計算し、0〜1の数値として返してください。

備考：
- `self.model.eval()`で評価モードに切り替え、`torch.no_grad()`で勾配の計算を止めています（あまり気にしなくていい）。これによりモデルのパラメータが更新されません。
- 最後のバッチは枚数が少ない場合があるため、バッチごとの精度の単純平均にはしないでください。

実装後，改めて学習済みモデルを渡してtrainerを作り直し、`trainer.evaluate(test_loader)`を実行してください。

In [ ]:
# ここにコードを書く
trainer = ...

### 演習：混同行列を作る

テスト画像全体について、正解クラスを行、予測クラスを列とする10×10の混同行列（confusion matrix）を作成してください。要素には、その組み合わせになった画像の枚数を入れます。

1. 混同行列をヒートマップとして描き、縦軸を正解、横軸を予測とし、各軸にクラス名を表示してください。
2. 全要素の合計がテスト画像の総数と一致することを確認してください。
3. 対角成分の合計を全要素の合計で割った値が、`evaluate`の分類精度と一致することを確認してください。

In [ ]:
# ここにコードを書く

## 5. モデルの保存と読み込み

学習したモデルのパラメータを`models/day2-cifar10.pt`に保存します。このファイルは次のNotebookでも使用します。


In [ ]:
from jkj1a.classification import save_classifier

model_path = "models/day2-cifar10.pt"
# 学習した重みを指定したファイルに保存する
save_classifier(model, path=model_path)


`load_classifier`で保存したパラメータを別のモデルに読み込みます。同じ画像を入力し、保存前と読み込み後の出力が一致することを確認してください。


In [ ]:
def load_classifier(path, device):
    # 保存時と同じ構造のモデルを作る
    model = CIFARClassifier().to(device)

    # ファイルからパラメータを読み込み、モデルに設定する
    state_dict = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)

    # 評価モードに切り替える
    model.eval()
    return model


In [ ]:
loaded_model = load_classifier(path=model_path, device=device)

# 同じ画像を、保存前のモデルと読み込んだモデルに入力する
x, y = next(iter(test_loader))
x = x.to(device)
model.eval()

with torch.no_grad():
    original_logits = model(x)
    loaded_logits = loaded_model(x)

# 浮動小数点の計算誤差を許容して比較する
print("出力が一致:", torch.allclose(original_logits, loaded_logits))
